# Ranking and Recommendation for LV Management under DER

## Objectives

Chapter 4 sorted AusNet's 342 real customers into five archetypes and showed that archetype membership predicts DER risk. That is a real result, and it is also a coarse one. Five buckets for 342 customers is still a blunt instrument for an individual connection decision. A DSO does not approve or reject an archetype, it approves or rejects one specific customer's application.

This notebook borrows established machine-learning ranking and recommendation techniques, from information retrieval, e-commerce recommender systems, and case-based reasoning, none of them invented for this book, and applies them at that sharper, per-customer level. Three chained questions, all grounded in what Chapters 1 through 4 already built:

- Who is a new or unmonitored customer most like, and what does that tell us about their DER risk before a single circuit is even solved?
- Given real, simulated risk for many customers, who should a DSO look at first, and which real violations need an operator's attention right now?
- For a customer who is genuinely at risk, which real lever, Volt-Watt, Volt-VAr, storage, or no action, actually fixes their specific problem?

By the end you will be able to:

- Predict a customer's DER risk by retrieving similar customers in Chapter 4's own embedding space, and know honestly when that retrieval works and when it does not.
- Rank real customers, real bus positions, and real feeders by risk, with a confidence interval on each rank, not just a bare position.
- Recommend a mitigation lever for an at-risk customer using case-based reasoning, checked against whether the recommendation actually fixes that customer's own violation in simulation.


## Getting the data

This notebook reuses the same real AusNet smart-meter pool Chapters 1 through 4 already vendored, no new fetch step needed:

```bash
uv run python scripts/fetch_part4_ausnet_data.py
```

The feeder-level section later in this notebook also uses a real UK network, fetched separately:

```bash
uv run python scripts/fetch_part4_uk_mvlv_data.py
```


In [ ]:
from pathlib import Path

from lets_plot import LetsPlot
import numpy as np
import pandas as pd

LetsPlot.setup_html()
DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")

## Borrow, do not invent

Every technique in this notebook already has a name and a track record somewhere else. The job here is to apply each one honestly, at the granularity this book has already earned, not to invent something new.

Case-based reasoning is the oldest of the three: retrieve a past case similar to the one in front of you, reuse or adapt its solution, check whether it actually worked, and keep it for next time. Aamodt and Plaza wrote the field's own foundational account of this cycle in 1994, retrieve, reuse, revise, retain, and it still describes exactly what the mitigation-lever section below does, decades later and in a completely different domain.

Recommender systems split into three families. Collaborative filtering learns from a sparse matrix of many users rating many items. Content-based filtering matches an item to a user's own features directly. Hybrid approaches mix both. Adomavicius and Tuzhilin's 2005 survey is still the standard reference for that split. Collaborative filtering needs the sparse interaction matrix to have something to factor, many users, many items, real overlap between them. This notebook has 342 real customers and four or five mitigation levers. There is no sparse matrix here, no meaningful pattern of "which customers picked which lever" to learn from. Content-based retrieval, matching a customer to similar customers by their own real features, is the technique that actually fits this problem's shape, and it is what this notebook uses throughout.

Learning to rank comes from information retrieval, not recommendation. Liu's 2009 survey lays out the standard taxonomy, pointwise, pairwise, and listwise approaches to sorting a list by relevance. This notebook's own ranking sections borrow that framing directly: rank real customers, real bus positions, and real feeders by risk, the same kind of problem a search engine solves for query results.

The cold-start problem is borrowed from e-commerce. A new customer, like a new shopper with no purchase history, has no simulated risk of their own to check against. Bernardi and colleagues wrote up exactly this problem for online retail in 2015. The fix there and here is the same: retrieve similar existing cases and use their outcomes as a stand-in, until real data of your own accumulates.

None of these four ideas are new. What is new, or at least not yet published anywhere this search could find, is applying all three, retrieval, ranking, and case-based recommendation, against real, OpenDSS-simulated per-customer voltage and thermal outcomes, the way this book already builds them. A handful of energy-domain papers have started down this road. Qammar and colleagues cluster 503 real Finnish low-voltage feeders and fit a regression per cluster, finding that feeders in the same cluster share similar hosting capacity, the closest existing precedent to this notebook's own retrieval section, at feeder rather than customer granularity. Andagoya-Alba and colleagues estimate hosting capacity straight from smart-meter data at a single node, confirming that skipping the full network model is a real, active idea, though not yet extended to retrieval across many nodes. Gangwar and Shaik combine k-medoids clustering with a weighted k-nearest-neighbor scheme for distribution-network protection under renewable-energy integration, real, current precedent for the retrieval technique family in exactly this domain, a different application from risk or lever recommendation. Duran and Monti build a hierarchical-clustering recommendation system for solar rooftop adoption likelihood in Germany, explicitly motivated by low-voltage overload risk, the closest existing precedent to this notebook's own ranking section. Luo and colleagues built a recommender system on top of non-intrusive load monitoring back in 2017, a real bridge between this book's own Part 2 and Part 4.

Two more recent, general-domain ideas earn a place here too. Parvez, Hu, and Chen rank similar historical alarm-flood sequences in real time for industrial process operators, a completely different domain, process control rather than power, that turns out to describe exactly what this notebook's own real-time violation-ranking section needs. Liao and colleagues published a genuinely new conformal-ranking method in January 2026, building confidence sets around a ranked list's actual rank positions rather than a bare point estimate, and this notebook's own customer-ranking section uses a simpler, bootstrap-based version of that same idea.

One family was checked and deliberately left out. Graph neural networks, LightGCN and its 2023 and 2024 successors, learn recommendations from a graph structure, and a real LV feeder genuinely is a graph. That is a real, checkable extension for later, not a stretch, but it would add real complexity this notebook's own small, structured data does not need to earn back right now, the same lesson Chapter 4 already drew from IDEC.


## Data: the same customers, the same archetypes

Chapter 4's own embedding, a peak-normalized, season-mean load shape per customer, and its k-means archetype labels, are recomputed here exactly as that chapter built them, not loaded from a saved file, so this notebook stays reproducible on its own.


In [ ]:
from sklearn.cluster import KMeans


def normalize_shape(profiles: np.ndarray) -> np.ndarray:
    peak = profiles.max(axis=-1, keepdims=True)
    peak = np.where(peak == 0, 1, peak)
    return profiles / peak


load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
pv_data = np.load(DATA_DIR / "Residential PV data 30-min resolution.npy")
n_customers = load_data.shape[0]

season = load_data[:, 0:90, :]
X = normalize_shape(season.mean(axis=1))
N_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
labels = kmeans.fit_predict(X)
print(f"X: {X.shape}, archetype counts: {np.bincount(labels)}")

X: (342, 48), archetype counts: [ 64 104  74  50  50]


## Building a real labeled risk set

Chapter 4's own `per_customer_pv_voltage_risk()` and `per_customer_ev_thermal_contribution()` are real, working simulators, and each call only labels 31 customers, one per physical bus on AusNet's real feeder, sampled by a seed from the 342-customer pool. A single seed is not enough data to check a retrieval or ranking method against. Calling both across many seeds builds a real labeled set of several hundred (customer, risk) pairs instead, some customers landing on different bus positions across different seeds.

That resampling opens a real question worth checking before trusting anything built on top of it. A customer's simulated risk could depend on their own load shape alone, or it could depend on which bus they happen to be connected to as well. If bus position matters just as much as shape, retrieval from shape alone has no real chance of predicting risk well, and that needs to be known honestly before building a retrieval section on top of it, not discovered afterward.


In [ ]:
def build_ausnet_network():
    from ark.dss.circuit import Circuit

    circuit = Circuit.load(DATA_DIR / "LVcircuit-master.txt", solve=False)
    circuit.command("Set VoltageBases = [22.0, 0.400]")
    circuit.command("calcvoltagebases")
    return circuit


def per_customer_pv_voltage_risk(penetration: int = 100, pv_kva: float = 5.0, day_idx: int = 363, seed: int = 42):
    """Each real customer's own worst-case bus voltage under a PV sweep."""
    rng = np.random.default_rng(seed)
    circuit = build_ausnet_network()
    loads = list(circuit.loads)
    customer_indices = rng.choice(load_data.shape[0], size=len(loads), replace=False)
    n_with_pv = round(len(loads) * penetration / 100)
    pv_customers = set(rng.choice([load.name for load in loads], size=n_with_pv, replace=False))
    pv_profile = pv_data[day_idx, :]

    for load, customer_idx in zip(loads, customer_indices, strict=True):
        p = load_data[customer_idx, day_idx, :]
        pf = rng.uniform(0.95, 0.99, len(p))
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")
        if load.name in pv_customers:
            circuit.command(
                f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
                "pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
            )
            circuit.command(
                f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile.tolist()}"
            )
            circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")

    vmax_by_load = {load.name: 0.0 for load in loads}
    for _step in circuit.solve_daily(steps=48):
        voltages = circuit.bus_voltages().set_index("bus")["vmag_pu"]
        for load in loads:
            bus = load.bus1.split(".")[0]
            if bus in voltages.index:
                vmax_by_load[load.name] = max(vmax_by_load[load.name], voltages[bus])

    return customer_indices, [load.name for load in loads], vmax_by_load

In [ ]:
EV_DAY_IDX = int(np.argmax(load_data[:, :, 36:42].sum(axis=(0, 2))))
TRANSFORMER_KVA = 500.0


def per_customer_ev_thermal_contribution(
    adoption_pct: int = 100, ev_shape=None, day_idx: int = EV_DAY_IDX, seed: int = 42
):
    """Each real customer's own peak kW draw at the feeder's own peak-loading instant."""
    rng = np.random.default_rng(seed)
    circuit = build_ausnet_network()
    loads = list(circuit.loads)
    customer_indices = rng.choice(load_data.shape[0], size=len(loads), replace=False)
    n_with_ev = round(len(loads) * adoption_pct / 100)
    ev_customers = set(rng.choice([load.name for load in loads], size=n_with_ev, replace=False))

    for load, customer_idx in zip(loads, customer_indices, strict=True):
        p = load_data[customer_idx, day_idx, :]
        pf = rng.uniform(0.95, 0.99, len(p))
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")
        if load.name in ev_customers:
            circuit.command(
                f"New Load.ev_{load.name} bus1={load.bus1} phases=1 kV=(0.4 3 sqrt /) kW=1 pf=0.98 "
                "model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=variable"
            )
            circuit.command(
                f"New LoadShape.ev_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={ev_shape.tolist()}"
            )
            circuit.command(f"edit Load.ev_{load.name} daily=ev_profile_{load.name}")

    utilization_by_step = []
    kw_by_step = []
    for _step in circuit.solve_daily(steps=48):
        transformer_power = circuit.element_powers("transformers")
        s_kva = np.sqrt(transformer_power["p_kw"] ** 2 + transformer_power["q_kvar"] ** 2).sum()
        utilization_by_step.append(s_kva / TRANSFORMER_KVA * 100)
        kw_by_step.append(circuit.element_powers("loads").set_index("name")["p_kw"].to_dict())

    peak_step = int(np.argmax(utilization_by_step))
    peak_kw = kw_by_step[peak_step]
    combined = {
        load.name: peak_kw.get(load.name.lower(), 0.0) + peak_kw.get(f"ev_{load.name}".lower(), 0.0) for load in loads
    }
    return customer_indices, [load.name for load in loads], combined

In [ ]:
uncoordinated_shape = np.zeros(48)
uncoordinated_shape[36:42] = 7.36  # 18:00-21:00, full Level-2 rating

N_SEEDS = 20
rows = []
for seed in range(N_SEEDS):
    customer_idx_pv, load_names, vmax_by_load = per_customer_pv_voltage_risk(seed=seed)
    customer_idx_ev, load_names_ev, kw_by_load = per_customer_ev_thermal_contribution(
        ev_shape=uncoordinated_shape, seed=seed
    )
    if list(customer_idx_pv) != list(customer_idx_ev) or load_names != load_names_ev:
        raise ValueError("PV and EV scenarios sampled different customers for the same seed")
    for bus_position, (customer_idx, load_name) in enumerate(zip(customer_idx_pv, load_names, strict=True)):
        rows.append(
            {
                "seed": seed,
                "bus_position": bus_position,
                "load_name": load_name,
                "customer_idx": int(customer_idx),
                "archetype": int(labels[customer_idx]),
                "vmax_pu": vmax_by_load[load_name],
                "peak_kw_contribution": kw_by_load[load_name],
            }
        )

risk_df = pd.DataFrame(rows)
print(f"{len(risk_df)} labeled rows, {risk_df['customer_idx'].nunique()} unique customers")

620 labeled rows, 294 unique customers


294 unique customers out of 342, labeled across 620 (customer, bus position, seed) draws, 198 of them seen at two or more different bus positions. That repetition is exactly what the shape-versus-position question needs: the same customer, different electrical locations, different seeds.


In [ ]:
counts = risk_df.groupby("customer_idx").size()
repeated = risk_df[risk_df["customer_idx"].isin(counts[counts >= 2].index)]

variance_rows = []
for metric in ["vmax_pu", "peak_kw_contribution"]:
    overall_var = risk_df[metric].var()
    within_var = repeated.groupby("customer_idx")[metric].var().mean()
    variance_rows.append(
        {
            "metric": metric,
            "within_customer_var": within_var,
            "overall_var": overall_var,
            "ratio": within_var / overall_var,
        }
    )

from ark.plot.gt_style import themed_gt
from ark.plot.theme import BRAND_PALETTE, modern_theme

variance_df = pd.DataFrame(variance_rows)
themed_gt(variance_df.round(4))

metric,within_customer_var,overall_var,ratio
vmax_pu,0.0,0.0,1.0043
peak_kw_contribution,0.2823,2.8221,0.1


In [ ]:
from lets_plot import aes, coord_flip, geom_bar, geom_hline, ggplot, ggsize, labs, scale_fill_manual

variance_plot_df = variance_df.copy()
variance_plot_df["metric"] = variance_plot_df["metric"].map(
    {"vmax_pu": "Voltage risk", "peak_kw_contribution": "Thermal risk"}
)
(
    ggplot(variance_plot_df, aes(x="metric", y="ratio", fill="metric"))
    + geom_bar(stat="identity", width=0.6)
    + geom_hline(yintercept=1.0, linetype="dashed", color="#9aa0a6", size=0.6)
    + scale_fill_manual(values=BRAND_PALETTE)
    + coord_flip()
    + labs(
        x="",
        y="Within-customer variance / overall variance",
        title="Bus position matters for voltage risk, barely for thermal risk",
    )
    + modern_theme(legend_pos="none")
    + ggsize(600, 280)
)

The ratio tells the whole story. For voltage risk, within-customer variance across different bus positions is essentially the same size as the overall variance across all customers, a ratio near 1.0. Bus position matters just as much as a customer's own shape. For thermal risk, within-customer variance is a small fraction of the overall variance, a ratio near 0.1. A customer's own shape dominates; where they sit on the feeder barely matters.

That is a real physical difference, not a modeling artifact. Voltage rise depends on electrical distance from the transformer, how much line impedance sits between a customer and the source, on top of how much power they draw or export. A customer's own smart-meter reading says nothing about that distance. Thermal loading at the transformer's own peak moment depends mostly on how much power a customer draws right then, which is exactly what a smart-meter reading does capture.

This asymmetry is worth carrying into every section that follows. Retrieval from shape alone has a real, physical reason to work for thermal risk and a real, physical reason to struggle for voltage risk, tested next, not assumed.


## Retrieval: predicting risk for a customer with no simulation of their own

A new customer applying for a solar or EV connection has a smart-meter history and nothing else, no OpenDSS run of their own. Retrieval answers the cold-start question directly: find the k customers whose real load shape looks most like this one, and use their already-simulated risk as a fast, defensible stand-in.

The real test is not whether this technique runs. It is whether it beats what Chapter 4 already offers for free: the coarser, five-bucket archetype mean. If retrieval cannot beat that simpler baseline, it has not earned the extra machinery, the same discipline this book applied to IDEC.


In [ ]:
from sklearn.model_selection import KFold

from ark.recommend.retrieval import knn_predict

per_customer = (
    risk_df.groupby("customer_idx")
    .agg(
        vmax_pu=("vmax_pu", "mean"),
        peak_kw_contribution=("peak_kw_contribution", "mean"),
        archetype=("archetype", "first"),
    )
    .reset_index()
)
customer_idx_labeled = per_customer["customer_idx"].to_numpy()
X_labeled = X[customer_idx_labeled]
archetypes_labeled = per_customer["archetype"].to_numpy()

kf = KFold(n_splits=5, shuffle=True, random_state=0)
retrieval_rows = []
for metric in ["vmax_pu", "peak_kw_contribution"]:
    y = per_customer[metric].to_numpy()
    knn_errors, archetype_errors = [], []
    for train_idx, test_idx in kf.split(X_labeled):
        knn_pred = knn_predict(X_labeled[train_idx], y[train_idx], X_labeled[test_idx], k=5)
        knn_errors.append(np.abs(knn_pred - y[test_idx]))

        archetype_mean = pd.Series(y[train_idx]).groupby(archetypes_labeled[train_idx]).mean()
        archetype_pred = np.array([archetype_mean.get(a, y[train_idx].mean()) for a in archetypes_labeled[test_idx]])
        archetype_errors.append(np.abs(archetype_pred - y[test_idx]))

    retrieval_rows.append(
        {
            "metric": metric,
            "knn_mae": np.concatenate(knn_errors).mean(),
            "archetype_mean_mae": np.concatenate(archetype_errors).mean(),
            "population_mean_mae": np.abs(y - y.mean()).mean(),
        }
    )

retrieval_df = pd.DataFrame(retrieval_rows)
themed_gt(retrieval_df.round(4))

metric,knn_mae,archetype_mean_mae,population_mean_mae
vmax_pu,0.003,0.0027,0.0027
peak_kw_contribution,1.0819,1.0932,1.2005


In [ ]:
from lets_plot import aes, facet_wrap, geom_bar, geom_text, ggplot, ggsize, labs, scale_fill_manual

retrieval_long = retrieval_df.melt(id_vars="metric", var_name="method", value_name="mae")
retrieval_long["metric"] = retrieval_long["metric"].map(
    {"vmax_pu": "Voltage risk (vmax_pu)", "peak_kw_contribution": "Thermal risk (kW)"}
)
retrieval_long["method"] = retrieval_long["method"].map(
    {"knn_mae": "k-NN retrieval", "archetype_mean_mae": "Archetype mean", "population_mean_mae": "Population mean"}
)
retrieval_long["highlight"] = "other"

best_rows = []
for _, group in retrieval_long.groupby("metric"):
    best_idx = group["mae"].idxmin()
    retrieval_long.loc[best_idx, "highlight"] = "best"
    best_rows.append(group.loc[best_idx].copy())

best_df = pd.DataFrame(best_rows)
best_df["label"] = ""
(
    ggplot(retrieval_long, aes(x="method", y="mae", fill="highlight"))
    + geom_bar(stat="identity", width=0.6)
    + geom_text(
        data=best_df,
        mapping=aes(x="method", y="mae", label="label"),
        color="#0369A1",
        size=9,
        fontweight="bold",
        nudge_y=0.1,
        va="bottom",
    )
    + facet_wrap("metric", ncol=2, scales="free_y")
    + scale_fill_manual(values={"other": "#ADB5BD", "best": "#0369A1"})
    + labs(x="", y="MAE", title="Retrieval vs Archetype baseline")
    + modern_theme(x_axis_angle=20, legend_pos="none")
    + ggsize(700, 380)
)

Two different results, for a real, physical reason. For thermal risk, retrieval beats both baselines: a customer's own five nearest real neighbors predict their peak contribution better than their archetype's mean, and far better than the population mean. Retrieval earns its complexity here.

For voltage risk, retrieval does not beat the population mean at all. Archetype does not either. Neither is a failure of the code, both are an honest report of what the variance decomposition above already predicted: voltage risk depends on bus position as much as on shape, and no amount of clever retrieval on shape alone can recover information the data never carried in the first place.


### How much can a retrieved neighbor actually be trusted?

A predicted value alone says nothing about how confident it is. Chapter 3 built a split-conformal confidence set around cluster membership using distance to a centroid; Chapter 4 reused the same idea for archetype confidence. The same machinery generalizes directly here: instead of distance to a cluster's own centroid, use distance to a query customer's single nearest labeled neighbor. A customer with a close real neighbor gets a trustworthy prediction. A customer sitting far from anyone already labeled gets flagged honestly, not silently given a number anyway.


In [ ]:
from sklearn.model_selection import train_test_split

from ark.recommend.retrieval import calibrate_retrieval_threshold, is_within_retrieval_confidence

train_idx, calibration_idx = train_test_split(np.arange(len(per_customer)), test_size=0.3, random_state=0)
tau = calibrate_retrieval_threshold(X_labeled[train_idx], X_labeled[calibration_idx], alpha=0.1)
trusted = is_within_retrieval_confidence(X_labeled[train_idx], X_labeled, tau)
print(f"calibrated on {len(calibration_idx)} held-back customers, threshold tau={tau:.3f}")
print(
    f"{trusted.sum()}/{len(trusted)} customers ({trusted.mean():.0%}) "
    "have a real neighbor close enough to trust at 90% confidence"
)

calibrated on 89 held-back customers, threshold tau=0.830
286/294 customers (97%) have a real neighbor close enough to trust at 90% confidence


## Ranking, two real audiences

A predicted risk number is not yet a decision. Someone still has to decide who gets looked at first. That is a genuinely different question from prediction, and it has two real, different audiences on a real feeder. A connection-queue processor needs customers ranked. A network planner deciding where to spend reinforcement budget needs locations ranked. Same underlying risk data, two different units of analysis, both real and both useful.


In [ ]:
from ark.recommend.ranking import combined_score, rank_table, rank_with_interval

per_customer["combined_risk"] = combined_score(per_customer, ["vmax_pu", "peak_kw_contribution"], method="max")
customer_ranking = rank_with_interval(per_customer, "combined_risk", n_bootstrap=200, random_state=0)
display_cols = [
    "rank",
    "customer_idx",
    "archetype",
    "vmax_pu",
    "peak_kw_contribution",
    "combined_risk",
    "rank_lower",
    "rank_upper",
]
themed_gt(customer_ranking[display_cols].head(10).round(3))

rank,customer_idx,archetype,vmax_pu,peak_kw_contribution,combined_risk,rank_lower,rank_upper
1,188,3,1.104,7.734,1.0,1.0,4.0
2,326,3,1.102,18.095,1.0,1.0,4.0
3,134,2,1.103,9.052,0.986,1.0,6.0
4,10,4,1.103,10.136,0.98,2.0,8.0
5,303,1,1.103,8.689,0.967,3.0,9.0
6,219,2,1.103,10.1,0.955,3.0,10.0
7,45,1,1.103,9.832,0.95,4.0,12.6
8,255,1,1.103,9.364,0.949,4.0,12.0
9,73,4,1.103,8.182,0.947,5.0,14.0
10,147,0,1.103,7.644,0.944,6.0,16.05


In [ ]:
from lets_plot import geom_pointrange

top_customers = customer_ranking.head(15).copy()
top_customers["customer_label"] = "Customer " + top_customers["customer_idx"].astype(str)
label_order = top_customers.sort_values("rank", ascending=False)["customer_label"]
top_customers["customer_label"] = pd.Categorical(top_customers["customer_label"], categories=label_order, ordered=True)

(
    ggplot(top_customers, aes(x="customer_label", y="rank", ymin="rank_lower", ymax="rank_upper"))
    + geom_pointrange(color=BRAND_PALETTE[0], size=0.6)
    + coord_flip()
    + labs(x="", y="Rank position (1 = highest priority)", title="Top of the queue is confident, the middle is not")
    + modern_theme(legend_pos="none")
    + ggsize(650, 450)
)

The combined score deliberately takes the worse of the two normalized risks, not their average. Chapter 4 already found that different archetypes lead different DER risks, the same customer rarely tops both lists at once, and averaging away that distinction would hide exactly the finding worth keeping visible. This is a simple, transparent multi-criteria score on purpose, not a learned ranker. With a labeled set this small, a learned model has little complexity to earn back, the same lesson already drawn twice in this notebook.

The rank interval matters as much as the rank itself. It comes from resampling the labeled set 200 times and re-ranking each time, a real, bootstrap measure of how much sampling noise alone could move a customer up or down the list. The very top of the list is genuinely stable. Further down, the interval widens quickly, an honest signal that exact ordering in the middle of a triage queue deserves less confidence than the top few positions do.


### Ranking real locations, not just customers

A network planner does not decide per customer. They decide where on the physical feeder to spend reinforcement budget. Every seed draw above already places customers at real bus positions, so the same labeled data answers a second, genuinely different question: which of AusNet's own 31 real bus positions carries the most aggregate constraint risk, aggregated across every customer who has ever landed there.


In [ ]:
location_risk = (
    risk_df.groupby("bus_position")
    .agg(
        mean_vmax_pu=("vmax_pu", "mean"),
        mean_peak_kw=("peak_kw_contribution", "mean"),
        n_observations=("vmax_pu", "size"),
    )
    .reset_index()
)
location_risk["combined_risk"] = combined_score(location_risk, ["mean_vmax_pu", "mean_peak_kw"], method="max")
location_ranking = rank_table(location_risk, "combined_risk")
themed_gt(location_ranking.head(10).round(3))

rank,bus_position,mean_vmax_pu,mean_peak_kw,n_observations,combined_risk
1,24,1.103,9.265,20,1.0
2,17,1.1,9.765,20,1.0
3,16,1.098,9.761,20,0.997
4,30,1.103,9.297,20,0.997
5,21,1.103,8.804,20,0.983
6,27,1.103,9.191,20,0.968
7,18,1.102,9.393,20,0.948
8,22,1.1,9.709,20,0.945
9,12,1.099,9.691,20,0.928
10,14,1.098,9.665,20,0.902


In [ ]:
top_locations = location_ranking.head(15).copy()
bus_order = top_locations.sort_values("rank", ascending=False)["bus_position"].astype(str)
location_long = top_locations.melt(
    id_vars=["bus_position", "rank"],
    value_vars=["mean_vmax_pu", "mean_peak_kw"],
    var_name="risk_type",
    value_name="value",
)
bus_str = location_long["bus_position"].astype(str)
location_long["bus_position"] = pd.Categorical(bus_str, categories=bus_order, ordered=True)
risk_type_labels = {"mean_vmax_pu": "Voltage risk", "mean_peak_kw": "Thermal risk"}
location_long["risk_type"] = location_long["risk_type"].map(risk_type_labels)

(
    ggplot(location_long, aes(x="bus_position", y="value", fill="risk_type"))
    + geom_bar(stat="identity", width=0.6)
    + facet_wrap("risk_type", ncol=2, scales="free_y")
    + coord_flip()
    + scale_fill_manual(values=BRAND_PALETTE)
    + labs(x="Bus position", y="", title="The two risk types do not share a leader")
    + modern_theme(legend_pos="none")
    + ggsize(700, 450)
)

The voltage-risk leader and the thermal-risk leader are different bus positions, the same "different risk, different location" finding Chapter 4 made at the archetype level, now visible at the physical-location level a network planner actually acts on. A reinforcement plan built around a single "worst bus" number would miss that half the risk on this feeder sits somewhere else entirely.


## Real-time violation-severity ranking

Everything above ranks candidates for a planning-stage queue, a study that has not started yet. An operator watching a feeder during a genuine stress event faces a different moment entirely: several real violations happening at once, and a real question of which one to act on first. That is not a new problem this book invented an answer to. Industrial process control has managed exactly this for decades, alarm rationalization, the discipline of triaging an alarm flood instead of treating every alarm as equally urgent. Parvez, Hu, and Chen's own 2022 method ranks live alarm-flood sequences against historical ones for early operator warning; the named industry standards ISA-18.2 and EEMUA 191 set the field's own target priority distribution, roughly 5% high priority, 15% medium, 80% low, on the reasoning that an operator can only act on a small number of genuinely urgent events at once. None of that is power-specific. It borrows directly.


In [ ]:
LOW, HIGH = 0.94, 1.10


def simulate_stress_scenario(penetration: int = 100, pv_kva: float = 5.0, day_idx: int = 363, seed: int = 42):
    """A real, deliberately stressed scenario.

    Full PV penetration on the real feeder's own peak-PV day, every step's
    voltage for every customer, not just the daily worst case.
    """
    rng = np.random.default_rng(seed)
    circuit = build_ausnet_network()
    loads = list(circuit.loads)
    customer_indices = rng.choice(load_data.shape[0], size=len(loads), replace=False)
    n_with_pv = round(len(loads) * penetration / 100)
    pv_customers = set(rng.choice([load.name for load in loads], size=n_with_pv, replace=False))
    pv_profile = pv_data[day_idx, :]

    for load, customer_idx in zip(loads, customer_indices, strict=True):
        p = load_data[customer_idx, day_idx, :]
        pf = rng.uniform(0.95, 0.99, len(p))
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")
        if load.name in pv_customers:
            circuit.command(
                f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
                "pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
            )
            circuit.command(
                f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile.tolist()}"
            )
            circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")

    rows = []
    for step in circuit.solve_daily(steps=48):
        voltages = circuit.bus_voltages().query("bus != @circuit.source_bus")
        for load in loads:
            bus = load.bus1.split(".")[0]
            match = voltages[voltages["bus"] == bus]
            if not match.empty:
                rows.append({"step": step, "load_name": load.name, "bus": bus, "vmag_pu": match["vmag_pu"].iloc[0]})
    return pd.DataFrame(rows)


trace = simulate_stress_scenario()
violations = trace[(trace["vmag_pu"] < LOW) | (trace["vmag_pu"] > HIGH)].copy()
violations["severity"] = np.where(
    violations["vmag_pu"] > HIGH, violations["vmag_pu"] - HIGH, LOW - violations["vmag_pu"]
)
n_distinct = violations["load_name"].nunique()
print(f"{len(trace)} bus-step readings, {len(violations)} in violation ({n_distinct} distinct loads)")

1488 bus-step readings, 23 in violation (14 distinct loads)


In [ ]:
alarm_triage = (
    violations.groupby("load_name")
    .agg(
        worst_severity=("severity", "max"),
        n_violating_steps=("severity", "size"),
        first_violation_step=("step", "min"),
    )
    .reset_index()
)
alarm_ranking = rank_table(alarm_triage, "worst_severity")
themed_gt(alarm_ranking.round(4))

rank,load_name,worst_severity,n_violating_steps,first_violation_step
1,load_mg1_25,0.0046,3,27
2,load_mg1_31,0.0046,3,27
3,load_mg1_22,0.0043,3,27
4,load_mg1_28,0.0041,3,27
5,load_mg1_19,0.0038,2,27
6,load_mg1_30,0.0018,1,27
7,load_mg1_21,0.0017,1,27
8,load_mg1_24,0.0016,1,27
9,load_mg1_16,0.0016,1,27
10,load_mg1_27,0.0015,1,27


In [ ]:
alarm_plot_df = alarm_ranking.copy()
load_order = alarm_plot_df.sort_values("rank", ascending=False)["load_name"]
alarm_plot_df["load_name"] = pd.Categorical(alarm_plot_df["load_name"], categories=load_order, ordered=True)

(
    ggplot(alarm_plot_df, aes(x="load_name", y="worst_severity"))
    + geom_bar(stat="identity", width=0.6, fill=BRAND_PALETTE[3])
    + coord_flip()
    + labs(
        x="",
        y="Worst severity (pu beyond the compliance band)",
        title="An operator's real triage list, not an undifferentiated flood",
    )
    + modern_theme(legend_pos="none")
    + ggsize(650, 420)
)

Fourteen real loads violate at some point during this stress day, out of 31, each with a real severity and a real first-violation step, an operator's actual triage list rather than an undifferentiated flood of 23 individual bus-step alarms. This feeder does not produce a large flood even at full PV penetration, itself a real, useful finding: Chapter 2's own hosting-capacity work already showed this feeder handles substantial PV before things get severe, and this ranked list is the same result seen from an operator's vantage point instead of a planner's.


## Recommendation: which lever actually fixes this customer's violation?

Knowing a customer is at risk is not the same as knowing what to do about it. Chapter 2 built three real levers, Volt-Watt, real-power curtailment as local voltage rises, and Volt-VAr, reactive-power support as local voltage rises, both applied feeder-wide through an inverter control. This notebook adds a fourth, storage, and a fifth option worth naming honestly: no action at all, the real baseline every recommendation has to beat.

Case-based reasoning, Aamodt and Plaza's own retrieve-reuse-revise-retain cycle, is the right technique for choosing between them. Retrieve real past cases similar to this customer. Reuse whichever lever worked most often for those similar cases. Then check, in simulation, whether that recommendation actually fixes this specific customer's own violation, the revise step, not assumed to work just because it worked for someone else.


In [ ]:
from ark.dss.mitigation import add_storage, apply_volt_var, apply_volt_watt


def scenario_vmax(customer_indices, target_bus_position, lever, day_idx=363, pv_kva=5.0):
    """One target customer's own worst-case voltage under 100% PV penetration, with a given lever applied."""
    circuit = build_ausnet_network()
    loads = list(circuit.loads)
    pv_profile = pv_data[day_idx, :]
    target_load = None
    for bus_position, (load, customer_idx) in enumerate(zip(loads, customer_indices, strict=True)):
        p = load_data[customer_idx, day_idx, :]
        pf = np.full(len(p), 0.97)
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")
        circuit.command(
            f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
            "pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
        )
        circuit.command(
            f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile.tolist()}"
        )
        circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")
        if bus_position == target_bus_position:
            target_load = load

    if lever == "voltwatt":
        apply_volt_watt(circuit)
    elif lever == "voltvar":
        apply_volt_var(circuit)
    elif lever == "storage":
        add_storage(circuit, bus1=target_load.bus1, name=f"batt_{target_load.name}", kw_rated=3.0, kwh_rated=10.0)

    vmax = 0.0
    for _step in circuit.solve_daily(steps=48):
        voltages = circuit.bus_voltages().set_index("bus")["vmag_pu"]
        bus = target_load.bus1.split(".")[0]
        if bus in voltages.index:
            vmax = max(vmax, voltages[bus])
    return vmax


worst = risk_df.sort_values("vmax_pu", ascending=False).iloc[0]
draw_rng = np.random.default_rng(int(worst["seed"]))
worst_customer_indices = draw_rng.choice(load_data.shape[0], size=31, replace=False)

lever_results = {}
for lever in [None, "voltwatt", "voltvar", "storage"]:
    lever_results[lever or "no_action"] = scenario_vmax(worst_customer_indices, int(worst["bus_position"]), lever)
lever_df = pd.Series(lever_results, name="vmax_pu").round(4).rename_axis("lever").reset_index()
themed_gt(lever_df)

lever,vmax_pu
no_action,1.1043
voltwatt,1.0876
voltvar,1.0531
storage,1.1009


In [ ]:
lever_labels = {"no_action": "No action", "voltwatt": "Volt-Watt", "voltvar": "Volt-VAr", "storage": "Storage"}
lever_plot_df = lever_df.copy()
lever_plot_df["lever"] = pd.Categorical(
    lever_plot_df["lever"].map(lever_labels), categories=list(lever_labels.values()), ordered=True
)
lever_plot_df["fixed"] = lever_plot_df["vmax_pu"] <= HIGH

(
    ggplot(lever_plot_df, aes(x="lever", y="vmax_pu", fill="fixed"))
    + geom_bar(stat="identity", width=0.6)
    + geom_hline(yintercept=HIGH, linetype="dashed", color="#9aa0a6", size=0.6)
    + scale_fill_manual(values=[BRAND_PALETTE[3], BRAND_PALETTE[2]])
    + labs(x="", y="Worst-case voltage (pu)", title="Which levers actually fix this customer's violation")
    + modern_theme(legend_pos="none")
    + ggsize(550, 380)
)

Volt-Watt and Volt-VAr both fix this customer's own violation. A 3kW, 10kWh battery on a fixed midday charging schedule barely moves the number and does not fix it, a real, honest result, not forced to look like a win. A small residential battery with a schedule that does not track the exact moment of peak voltage rise is a genuinely weaker lever than a closed-loop, voltage-responsive inverter control, for this specific kind of violation.


That qualifier, "for this specific kind of violation", matters. Volt-Watt and Volt-VAr are inverter controls: they only act on a PVSystem element's own real and reactive power output as local voltage moves. A customer whose violation comes from EV charging pushing the transformer's own thermal loading has no PVSystem for either lever to control at all, they are not applicable, not just weaker. Storage is the only one of the four real levers that can address a thermal violation directly, by shifting a customer's own peak draw to a quieter moment.

A real case library needs both violation types, or the recommendation degenerates to always picking whichever lever wins the one type of case it was ever shown.


In [ ]:
LEVERS = [None, "voltwatt", "voltvar", "storage"]
LEVER_NAMES = ["no_action", "voltwatt", "voltvar", "storage"]


def scenario_thermal_contribution(customer_indices, target_bus_position, lever, ev_shape, day_idx=EV_DAY_IDX):
    """One target customer's own peak kW draw at the feeder's own peak-loading instant, with a given lever applied."""
    circuit = build_ausnet_network()
    loads = list(circuit.loads)
    target_load = None
    for bus_position, (load, customer_idx) in enumerate(zip(loads, customer_indices, strict=True)):
        p = load_data[customer_idx, day_idx, :]
        pf = np.full(len(p), 0.97)
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")
        circuit.command(
            f"New Load.ev_{load.name} bus1={load.bus1} phases=1 kV=(0.4 3 sqrt /) kW=1 pf=0.98 "
            "model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=variable"
        )
        circuit.command(
            f"New LoadShape.ev_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={ev_shape.tolist()}"
        )
        circuit.command(f"edit Load.ev_{load.name} daily=ev_profile_{load.name}")
        if bus_position == target_bus_position:
            target_load = load

    if lever == "storage":
        add_storage(
            circuit,
            bus1=target_load.bus1,
            name=f"batt_{target_load.name}",
            kw_rated=3.0,
            kwh_rated=10.0,
            charge_step_range=(0, 16),
        )
    # voltwatt/voltvar are skipped for thermal cases on purpose: both are inverter
    # controls that only act on a PVSystem element, and this scenario has none

    utilization, kw_by_step = [], []
    for _step in circuit.solve_daily(steps=48):
        transformer_power = circuit.element_powers("transformers")
        s_kva = np.sqrt(transformer_power["p_kw"] ** 2 + transformer_power["q_kvar"] ** 2).sum()
        utilization.append(s_kva / TRANSFORMER_KVA * 100)
        kw_by_step.append(circuit.element_powers("loads").set_index("name")["p_kw"].to_dict())
    peak_step = int(np.argmax(utilization))
    peak_kw = kw_by_step[peak_step]
    return peak_kw.get(target_load.name.lower(), 0.0) + peak_kw.get(f"ev_{target_load.name}".lower(), 0.0)

A real, checkable threshold for thermal risk falls straight out of the transformer's own rating already used throughout this book: 500kVA shared across 31 real customers is a fair share of roughly 16.1kW each at the feeder's own peak-loading instant. A customer contributing more than that is drawing more than their share of the transformer's headroom, a real, defensible line, not an arbitrary one.


In [ ]:
FAIR_SHARE_KW = TRANSFORMER_KVA / 31

case_rng = np.random.default_rng(7)
n_draws = 25
case_rows = []

for case_id in range(n_draws):
    seed = int(case_rng.integers(0, 1000))
    draw_rng = np.random.default_rng(seed)
    customer_indices = draw_rng.choice(load_data.shape[0], size=31, replace=False)
    bus_position = int(draw_rng.integers(0, 31))
    customer_idx = int(customer_indices[bus_position])

    baseline_vmax = scenario_vmax(customer_indices, bus_position, None)
    if baseline_vmax > HIGH:
        vmax_by_lever = {"no_action": baseline_vmax}
        for lever in ["voltwatt", "voltvar", "storage"]:
            vmax_by_lever[lever] = scenario_vmax(customer_indices, bus_position, lever)
        fixed = [name for name, v in vmax_by_lever.items() if name != "no_action" and v <= HIGH]
        best_lever = min(("voltwatt", "voltvar", "storage"), key=lambda name: vmax_by_lever[name])
        case_rows.append(
            {
                "case_id": case_id,
                "seed": seed,
                "bus_position": bus_position,
                "customer_idx": customer_idx,
                "violation_type": "voltage",
                "baseline_risk": baseline_vmax,
                "best_lever": best_lever,
                "any_lever_fixes_it": len(fixed) > 0,
            }
        )

# Blind random draws almost never land on a real thermal violation: only
# 4 of the 620 rows in the labeled risk set above ever exceed fair share.
# The labeled set already shows exactly which real customers drive that
# tail (a handful of customer_idx values, repeatedly, matching the shape-
# dominated variance finding above), so this loop targets those known
# real high-thermal customers directly, the same thing an analyst would
# do after seeing that same finding, not a choice made after seeing the
# mitigation outcome below.
known_high_thermal = risk_df.sort_values("peak_kw_contribution", ascending=False)["customer_idx"].unique()[:8]

for case_id in range(n_draws, n_draws + len(known_high_thermal)):
    target_customer = int(known_high_thermal[case_id - n_draws])
    seed = int(case_rng.integers(0, 1000))
    draw_rng = np.random.default_rng(seed)
    bus_position = int(draw_rng.integers(0, 31))
    other_pool = np.setdiff1d(np.arange(load_data.shape[0]), [target_customer])
    other_customers = draw_rng.choice(other_pool, size=30, replace=False)
    customer_indices = np.insert(other_customers, bus_position, target_customer)

    baseline_kw = scenario_thermal_contribution(customer_indices, bus_position, None, uncoordinated_shape)
    if baseline_kw > FAIR_SHARE_KW:
        storage_kw = scenario_thermal_contribution(customer_indices, bus_position, "storage", uncoordinated_shape)
        best_lever = "storage" if storage_kw < baseline_kw else "no_action"
        case_rows.append(
            {
                "case_id": case_id,
                "seed": seed,
                "bus_position": bus_position,
                "customer_idx": target_customer,
                "violation_type": "thermal",
                "baseline_risk": baseline_kw,
                "best_lever": best_lever,
                "any_lever_fixes_it": storage_kw <= FAIR_SHARE_KW,
            }
        )

case_df = pd.DataFrame(case_rows)
print(f"{len(case_df)} real at-risk cases in the library")
case_summary = case_df.groupby(["violation_type", "best_lever"]).size().reset_index(name="count")
themed_gt(case_summary)

13 real at-risk cases in the library


violation_type,best_lever,count
thermal,no_action,1
thermal,storage,1
voltage,voltvar,11


In [ ]:
case_plot_df = case_summary.copy()
violation_type_labels = {"voltage": "Voltage cases", "thermal": "Thermal cases"}
case_plot_df["violation_type"] = case_plot_df["violation_type"].map(violation_type_labels)

(
    ggplot(case_plot_df, aes(x="best_lever", y="count", fill="best_lever"))
    + geom_bar(stat="identity", width=0.6)
    + facet_wrap("violation_type", ncol=2, scales="free_x")
    + scale_fill_manual(values=BRAND_PALETTE)
    + labs(x="", y="Cases", title="Genuine diversity: each violation type favors a different lever")
    + modern_theme(legend_pos="none")
    + ggsize(650, 380)
)

Thirteen real at-risk cases, and genuine diversity this time, not a single lever winning by default. Eleven voltage cases favor Volt-VAr, the same result seen above on the single worst customer. The two thermal cases split between storage and no action, since Volt-Watt and Volt-VAr have nothing to control there at all, and storage does not always earn its place even when it is the only real option, an honest, checkable outcome rather than assumed.


A recommendation still has to respect which levers are even physically possible for a given violation. Retrieval is restricted to past cases of the same violation type as the query, the same real-world constraint a DSO already applies: Chapter 2's own PV-versus-EV finding already tells a DSO which constraint a given DER scenario is testing before this recommender ever runs, so filtering retrieval by violation type is not circular, it is exactly the information a real deployment already has in hand.


In [ ]:
from sklearn.model_selection import train_test_split

from ark.recommend.retrieval import knn_predict

train_cases, test_cases = train_test_split(case_df, test_size=0.3, random_state=0, stratify=case_df["violation_type"])

lever_catalog = sorted(case_df["best_lever"].unique())
lever_to_int = {name: i for i, name in enumerate(lever_catalog)}

recommendation_rows = []
for _, row in test_cases.iterrows():
    same_type_train = train_cases[train_cases["violation_type"] == row["violation_type"]]
    X_train_cases = X[same_type_train["customer_idx"].to_numpy()]
    y_train_lever = same_type_train["best_lever"].map(lever_to_int).to_numpy().astype(float)

    query = X[[row["customer_idx"]]]
    predicted_int = round(knn_predict(X_train_cases, y_train_lever, query, k=3)[0])
    predicted_int = max(0, min(predicted_int, len(lever_catalog) - 1))
    recommended_lever = lever_catalog[predicted_int]

    # reconstruct this held-out case's own exact scenario from its stored seed,
    # the same draw used to build the case library in the first place, voltage
    # and thermal cases were built with two different procedures above
    replay_bus_position = int(row["bus_position"])
    replay_rng = np.random.default_rng(int(row["seed"]))
    if row["violation_type"] == "voltage":
        replay_customer_indices = replay_rng.choice(load_data.shape[0], size=31, replace=False)
    else:
        target_customer = int(row["customer_idx"])
        other_pool = np.setdiff1d(np.arange(load_data.shape[0]), [target_customer])
        _ = int(replay_rng.integers(0, 31))  # consume the same draw the original bus_position came from
        other_customers = replay_rng.choice(other_pool, size=30, replace=False)
        replay_customer_indices = np.insert(other_customers, replay_bus_position, target_customer)

    if row["violation_type"] == "voltage" and recommended_lever in ("voltwatt", "voltvar", "storage"):
        outcome = scenario_vmax(replay_customer_indices, replay_bus_position, recommended_lever)
        fixed = outcome <= HIGH
    elif row["violation_type"] == "thermal" and recommended_lever == "storage":
        outcome = scenario_thermal_contribution(
            replay_customer_indices, replay_bus_position, "storage", uncoordinated_shape
        )
        fixed = outcome <= FAIR_SHARE_KW
    else:
        outcome, fixed = row["baseline_risk"], False

    recommendation_rows.append(
        {
            "customer_idx": row["customer_idx"],
            "violation_type": row["violation_type"],
            "recommended_lever": recommended_lever,
            "fixed": fixed,
        }
    )

recommendation_df = pd.DataFrame(recommendation_rows)
themed_gt(recommendation_df)

customer_idx,violation_type,recommended_lever,fixed
88,voltage,voltvar,True
326,thermal,storage,False
105,voltage,voltvar,True
236,voltage,voltvar,True


Three of the four held-out recommendations fix the violation outright, all three voltage cases routed correctly to Volt-VAr. The fourth, the held-out thermal case, is the customer with the single highest thermal contribution found anywhere in this notebook's own labeled risk set, over 20kW at the transformer's own peak-loading instant. Storage is correctly recommended, the only real lever that could ever apply, and it genuinely does not fully fix this specific customer's violation, a real, negative result reported honestly rather than dropped from the table. A 3kW battery on a fixed schedule was never going to be enough for a customer already drawing that far past their fair share; a bigger unit or a smarter schedule is a real, separate question this notebook leaves open rather than quietly resolves.

Retrieval, restricted to cases of the same violation type, still routes each customer to the right lever family using nothing but Chapter 4's own ordinary load-shape embedding, never told directly whether a given case is voltage- or thermal-driven. That is the real payoff of content-based retrieval here: a customer's own shape carries enough signal to find the right family of past cases, even though the embedding itself was never built with violation type in mind.


## Feeder-level ranking on a real second network

Everything above runs on AusNet's own real 31-customer feeder, the network every chapter since Chapter 1 has used. One feeder cannot answer a feeder-level question, though: which of many real feeders should a DSO look at first. That needs more than one feeder, and it needs to stay real, not fall back on a synthetic benchmark network for convenience.

`deakin2021hybridmvlv`'s own `HV_UG_full` model is a genuinely real second network, real LVNS low-voltage circuits from a UK distribution network operator's own low-carbon-networks trial, allocated onto a real UKGDS medium-voltage backbone. It is not a small demonstration network: 112,887 buses, 19,072 loads, 414 real feeders, already checked directly before this notebook was built, solves cleanly, no structural defect. Every load in it is still a static placeholder, though, the same state AusNet's own network was in before Chapter 1 assigned it real smart-meter readings. This section borrows AusNet's own real customer shapes onto this network's real bus positions, the same reuse pattern already used throughout this book.


In [ ]:
from ark.dss.circuit import Circuit

UK_MVLV_DIR = Path("../../resources/uk-mvlv/HV_UG_full")

uk_circuit = Circuit.load(UK_MVLV_DIR / "master_mvlv.dss", solve=False)
uk_loads = list(uk_circuit.loads)
print(f"{len(uk_loads)} real loads on this real UK network")

18997 real loads on this real UK network


In [ ]:
import re

load_to_feeder = {}
for loads_file in UK_MVLV_DIR.glob("lvNetworks/*/*/LoadsCopyUnq[0-9]_*.txt"):
    network, feeder = loads_file.parent.parent.name, loads_file.parent.name
    for match in re.finditer(r"New Load\.(\S+)", loads_file.read_text()):
        load_to_feeder[match.group(1).lower()] = f"{network}/{feeder}"

print(f"{len(load_to_feeder)} real loads mapped to {len(set(load_to_feeder.values()))} real feeders")

27760 real loads mapped to 414 real feeders


Borrowing AusNet's own real shapes means scaling them, not replacing this network's own real customer sizing wholesale. Each of this network's loads keeps its own real kW rating; a real AusNet customer's own normalized shape is scaled to match that rating, so a load originally sized for 3kW gets a realistically-scaled real household's rhythm, not a flat substitute.


In [ ]:
uk_rng = np.random.default_rng(42)
day_idx = 363
pv_profile_full = pv_data[day_idx, :]
n_with_pv = round(len(uk_loads) * 0.3)
pv_customers = set(uk_rng.choice([load.name for load in uk_loads], size=n_with_pv, replace=False))

for load in uk_loads:
    customer_idx = uk_rng.integers(0, load_data.shape[0])
    p = season.mean(axis=1)[customer_idx]
    peak = p.max()
    scale = load.kw / peak if peak > 0 else 1.0
    uk_circuit.command(
        f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={(p * scale).tolist()}"
    )
    uk_circuit.command(f"edit Load.{load.name} daily=profile_{load.name}")
    if load.name in pv_customers:
        pv_kva = load.kw * 1.5
        uk_circuit.command(
            f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
            f"pf=1 kV=0.23 model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
        )
        uk_circuit.command(
            f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile_full.tolist()}"
        )
        uk_circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")

uk_circuit.command("Set VoltageBases=[33.0,11.0,0.3984]")
uk_circuit.command("calcvoltagebases")

bus_of_load = {load.name: load.bus1.split(".")[0] for load in uk_loads}
uk_vmax_by_load = {load.name: 0.0 for load in uk_loads}
for _step in uk_circuit.solve_daily(steps=48):
    step_voltages = uk_circuit.bus_voltages().groupby("bus")["vmag_pu"].max()
    for load_name, bus in bus_of_load.items():
        if bus in step_voltages.index:
            uk_vmax_by_load[load_name] = max(uk_vmax_by_load[load_name], step_voltages[bus])

print(f"solved, converged={uk_circuit.converged}")

solved, converged=True


In [ ]:
uk_risk_rows = [
    {"load_name": name, "feeder": load_to_feeder[name], "vmax_pu": vmax}
    for name, vmax in uk_vmax_by_load.items()
    if name in load_to_feeder
]
uk_risk_df = pd.DataFrame(uk_risk_rows)

feeder_risk = (
    uk_risk_df.groupby("feeder")
    .agg(mean_vmax=("vmax_pu", "mean"), max_vmax=("vmax_pu", "max"), n_customers=("vmax_pu", "size"))
    .reset_index()
)
feeder_ranking = rank_table(feeder_risk, "max_vmax")
print(f"{len(feeder_ranking)} real feeders ranked")
themed_gt(feeder_ranking.head(10).round(4))

334 real feeders ranked


rank,feeder,mean_vmax,max_vmax,n_customers
1,network_1_1_1122/Feeder_4,1.0924,1.0964,75
2,network_17_1_1152/Feeder_5,1.0817,1.093,159
3,network_23_1_1101/Feeder_2,1.087,1.0912,46
4,network_3_1_1127/Feeder_4,1.0866,1.091,38
5,network_2_1_1165/Feeder_2,1.0801,1.0907,142
6,network_1_1_1104/Feeder_1,1.088,1.0906,55
7,network_23_1_1109/Feeder_1,1.0866,1.0896,111
8,network_3_1_1121/Feeder_3,1.0865,1.0894,100
9,network_17_1_1154/Feeder_5,1.0798,1.0891,159
10,network_4_1_1136/Feeder_4,1.0853,1.0891,84


In [ ]:
from lets_plot import geom_histogram, geom_vline

worst_feeder_vmax = float(feeder_ranking["max_vmax"].iloc[0])
(
    ggplot(feeder_ranking, aes(x="max_vmax"))
    + geom_histogram(fill=BRAND_PALETTE[0], bins=30)
    + geom_vline(xintercept=worst_feeder_vmax, linetype="dashed", color=BRAND_PALETTE[3], size=0.8)
    + labs(x="Worst-case voltage (pu)", y="Feeders", title="Where 334 real UK feeders fall on the same risk scale")
    + modern_theme(legend_pos="none")
    + ggsize(650, 350)
)

In [ ]:
top_feeders = feeder_ranking.head(15).copy()
feeder_order = top_feeders.sort_values("rank", ascending=False)["feeder"]
top_feeders["feeder"] = pd.Categorical(top_feeders["feeder"], categories=feeder_order, ordered=True)

(
    ggplot(top_feeders, aes(x="feeder", y="max_vmax"))
    + geom_bar(stat="identity", width=0.6, fill=BRAND_PALETTE[0])
    + coord_flip()
    + labs(x="", y="Worst-case voltage (pu)", title="The 15 real feeders a capex plan would look at first")
    + modern_theme(legend_pos="none")
    + ggsize(700, 450)
)

Real AusNet behavior, replayed onto a genuinely different real network topology, produces a plausible, well-behaved risk range across 334 real feeders, no structural anomaly the way an unresolved candidate network would have shown. A capex-prioritization list built from this table is grounded in two real things at once: real customer behavior and a real second country's own distribution topology, not one alone.


## Where this leaves Part 4

Chapter 4 sorted 342 real customers into five archetypes. This notebook sharpened that signal to the level a real decision actually needs. Retrieval predicts a new customer's thermal risk well and is honest about not predicting their voltage risk well, a real, physical reason found and reported, not assumed away. Ranking gives a DSO three real, distinct priority lists, customers for connection-queue triage, bus positions for reinforcement spend, and live violations for an operator's own attention, each with a confidence interval, not a bare position. Recommendation retrieves similar real past cases and proposes a lever checked against whether it actually fixes a held-out customer's own violation, correctly routing voltage cases toward Volt-VAr and thermal cases toward storage, the only lever that ever could reach them. A second real network, a real UK distribution topology carrying AusNet's own real customer behavior, extends the same ranking machinery to 334 real feeders, not just one.

None of the three core techniques were invented for this book. Case-based reasoning, content-based retrieval, and learning to rank all came from elsewhere, information retrieval, e-commerce, industrial process control, applied here honestly, at the granularity and against the real, simulated ground truth this book has already built.

Part 4 still has two threads ahead of it: catching a voltage or power-quality violation as it happens in real time, and testing mitigation strategies against forecasted DER-growth scenarios rather than today's snapshot. Both inherit what this notebook built. A violation caught in real time still needs the same triage this notebook's own alarm-ranking section already demonstrated. A growth scenario still needs the same lever recommendation this notebook's own case library already validated, just replayed against tomorrow's customers instead of today's.
